# RFSoC 4x2 – Pulse / Arbitrary Waveform Generator

**What this does:** plays up to **4 waveforms** (2 outputs × 2 slots) from the board's DACs, with
sample-accurate timing (1 sample = 101.7 ps).

**How to use it – 4 steps**
1. **Settings** – choose the length of the waveform window.
2. **Waveforms** – describe your pulses (height, delay, duration, shape; optional carrier or custom wave).
3. **Preview** – plot and check levels *before* touching the hardware.
4. **Play** – load the FPGA, upload, start / stop.

Steps 1-3 also run on a normal laptop with `DRY_RUN = True`.

**Hardware limits to remember**

| Item | Value |
|---|---|
| Outputs | `"DAC_A"` and `"DAC_B"` |
| Slots per output | 2 (`"z"` and `"x"`), same length for all |
| Max slot length | 26.67 µs (2¹⁸ samples) |
| Length granularity | 1.6276 ns (16 samples) – rounded automatically |
| Time resolution inside the window | 101.7 ps (1 sample) |
| Max amplitude | ±0.25 V peak ≈ −2 dBm (into 50 Ω); more gets clipped (you get a warning) |

In [ ]:
# Needs rfsoc_awg.py in the same folder as this notebook.
import matplotlib.pyplot as plt
from rfsoc_awg import *      # AWG, Pulse and the unit helpers ns, us, kHz, MHz, GHz

%matplotlib inline

## Step 1 – Settings

In [ ]:
DRY_RUN = False        # True = design/plot only, never touch the board (works on any PC)

# Length of ONE waveform slot. It is rounded to a multiple of 1.6276 ns; the real value is printed.
# (Max 26.67 us.)
if DRY_RUN:
    bitfile = None
else:
    from pynq import Overlay
    import xrfclk
    ol = Overlay("awg_adc.bit")                              # load the FPGA bit file
    xrfclk.set_ref_clks(lmk_freq=245.76, lmx_freq=491.52)    # program the board oscillators
    bitfile = ol                                             # AWG accepts a loaded Overlay (or a .bit path)

awg = AWG(window=10 * us, bitfile=bitfile, dry_run=DRY_RUN)

## Step 2 – Describe your waveforms

`awg.set_waveform(output, slot, what_to_play)` where `what_to_play` is a `Pulse`, a list of
`Pulse` objects (they are added together), or your own numpy array in volts.

`Pulse` options (all times in seconds – use `ns`, `us`):

| Argument | Meaning |
|---|---|
| `amp` **or** `power_dbm` **or** `vpp` | height: peak volts (signed, `amp=-0.1` inverts), dBm into 50 Ω, or volts peak-to-peak |
| `start` | delay: when the pulse begins, counted from the start of the slot (default 0) |
| `duration` | pulse length (default: until the end of the window) |
| `freq` | carrier frequency. **Default 0 = no carrier**, just an envelope pulse |
| `shape` | window/envelope: `"rect"`, `"gauss"`, `"cos"` (smooth bump) or `"flat"` (flat top with smooth edges, needs `rise=`) |
| `wave` | custom signal inside the pulse, e.g. `chirp(50*MHz, 70*MHz)`, your own `f(t, T)` or a numpy array. It is windowed by `start`, `duration` and `shape` |
| `phase_deg`, `phase_ref` | carrier phase (only matters with `freq > 0`) |

Both outputs and both slots share one clock, so `start` times are sample-accurate **between** channels too (101.7 ps).

### Pulse examples (copy / adapt – each call replaces what the slot played before)

In [ ]:
import numpy as np

# A) Delay between two channels: DAC_A fires at 1 us, DAC_B fires tau later.
tau = 350 * ns                                   # <- scan this
awg.set_waveform("DAC_A", "z", Pulse(amp=0.2, start=1 * us,         duration=100 * ns))
awg.set_waveform("DAC_B", "z", Pulse(amp=0.2, start=1 * us + tau,   duration=100 * ns))

# B) A sequence: 5 pulses, 1 us apart, first at 0.5 us (python list comprehension)
awg.set_waveform("DAC_A", "x", [Pulse(amp=0.15, start=0.5 * us + k * 1 * us, duration=50 * ns)
                                for k in range(5)])

# C) Sequence with different heights, widths and a negative pulse, e.g. pi/2 - wait - pi - wait - pi/2
t0, wait = 1 * us, 2 * us
awg.set_waveform("DAC_B", "x", [
    Pulse(amp=0.1,  start=t0,                  duration=100 * ns, shape="gauss"),
    Pulse(amp=0.2,  start=t0 + wait,           duration=200 * ns, shape="gauss"),
    Pulse(amp=-0.1, start=t0 + 2 * wait,       duration=100 * ns, shape="gauss"),
])

# D) Window a chirp: 50 -> 70 MHz that lasts exactly 8 us, starting at 1 us, with 200 ns smooth edges
#    (swap awg.set_waveform targets as needed - this one overwrites DAC_A slot z)
# awg.set_waveform("DAC_A", "z",
#                  Pulse(wave=chirp(50 * MHz, 70 * MHz), power_dbm=-10,
#                        start=1 * us, duration=8 * us, shape="flat", rise=200 * ns))

# E) Your own signal, windowed the same way. wave(t, T): t = time since pulse start, T = duration.
#    Here: a 30 MHz tone with a slow 1 MHz amplitude wobble, 3 us long, Gaussian window.
# my_wave = lambda t, T: np.sin(2 * np.pi * 30 * MHz * t) * np.cos(2 * np.pi * 1 * MHz * t)
# awg.set_waveform("DAC_A", "z", Pulse(wave=my_wave, amp=0.2, start=2 * us, duration=3 * us, shape="gauss"))
#    ...or a ready numpy array in [-1, 1] (one value per DAC sample); its length sets the duration:
# awg.set_waveform("DAC_A", "z", Pulse(wave=np.hanning(2000) * np.sin(np.arange(2000) / 4), amp=0.2, start=1 * us))

# F) Pulse with a carrier is still possible:  Pulse(freq=70 * MHz, power_dbm=-10, start=1 * us, duration=200 * ns)

# G) Empty a slot (outputs zero):  awg.set_waveform("DAC_B", "x", None)

## Step 3 – Preview (no hardware needed)
Always check this before playing. Red dotted lines = DAC full scale.
To inspect individual carrier cycles, zoom: `awg.plot(1*us, 1.05*us)`.

In [ ]:
awg.summary()   # peak level of every slot (+ clipping warning)
awg.plot()
plt.show()

## Step 4 – Play on the board
`connect()` resets the AWG logic of the already-loaded overlay (the bitstream and clocks were set up in Step 1; if you pass a `.bit` path instead of an `Overlay`, `connect()` loads it and programs the clocks itself).
`upload()` sends the waveforms to the board (repeat after every change).

In [ ]:
awg.connect()    # load bitstream, program clocks, reset the AWG logic
awg.upload()     # send waveforms + length to the board RAM
awg.start()      # enable the outputs

### Changed a pulse? Re-send and restart
Edit the `set_waveform(...)` calls above, then run this cell (no need to `connect()` again).

In [ ]:
awg.play()       # = stop -> upload -> start

### Stop

In [ ]:
awg.stop()

### Optional – check the board clocks
Expected: CPU ≈ 1200 MHz, FCLK0 ≈ 214 MHz, FCLK1-3 ≈ 100 MHz.

In [ ]:
if not DRY_RUN:
    print_clocks()

## Troubleshooting
* **`pynq`/`xrfclk` import error** – you are not on the board. Use `DRY_RUN = True`.
* **"Pulse ends at … but the window is only …"** – increase `window` in step 1 or shorten the pulse.
* **"Clipping!" warning** – requested level is above the DAC full scale (about −2 dBm). Lower `power_dbm`/`vpp`.
* **Absolute power looks off** – the dBm ↔ code calibration (`VPP_AT_FULL_SCALE` = 0.5 V in `rfsoc_awg.py`) comes from the
  original script; measure with a scope / spectrum analyser and update it if your output chain differs.
* **Nothing at the output** – did you call `awg.start()` after `awg.upload()`? Is the channel you use defined (outputs
  with no waveform are left disabled)?